In [4]:
# 🎓 Assignment-05: Spam Detection
# Ali-Salad | Bootcamp Submission

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# -----------------------------
# 1. Load Dataset
# -----------------------------
df = pd.read_csv("mail_l7_dataset.csv")   # file is in same folder as notebook
df = df.fillna("")   # handle missing values

# Encode labels: spam=0, ham=1
df["Category"] = df["Category"].map({"spam":0, "ham":1})

X = df["Message"]
y = df["Category"]

# -----------------------------
# 2. Train/Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -----------------------------
# 3. Text Feature Extraction
# -----------------------------
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# -----------------------------
# 4. Train Models (with balancing)
# -----------------------------
models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        class_weight="balanced", n_estimators=200, random_state=42
    ),
    "Naive Bayes": MultinomialNB()
}

results = {}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    results[name] = {
        "Accuracy": round(accuracy_score(y_test, y_pred), 2),
        "Precision": round(precision_score(y_test, y_pred), 2),
        "Recall": round(recall_score(y_test, y_pred), 2),
        "F1-Score": round(f1_score(y_test, y_pred), 2),
        "Confusion Matrix": confusion_matrix(y_test, y_pred)
    }

# -----------------------------
# 5. Print Results
# -----------------------------
for name, metrics in results.items():
    print(f"\n{name} Performance:")
    for metric, value in metrics.items():
        print(f"{metric} : {value}")

# -----------------------------
# 6. Sanity Checks
# -----------------------------
test_messages = [
    "Free entry in 2 a weekly competition!",
    "I will meet you at the cafe tomorrow",
    "Congratulations, you won a free ticket"
]

for msg in test_messages:
    vec = vectorizer.transform([msg])
    print(f"\nMessage: {msg}")
    for name, model in models.items():
        pred = model.predict(vec)[0]
        label = "Ham" if pred == 1 else "Spam"
        print(f"{name}: {label}")



Logistic Regression Performance:
Accuracy : 0.98
Precision : 0.99
Recall : 0.99
F1-Score : 0.99
Confusion Matrix : [[138  11]
 [ 13 953]]

Random Forest Performance:
Accuracy : 0.97
Precision : 0.97
Recall : 1.0
F1-Score : 0.98
Confusion Matrix : [[116  33]
 [  1 965]]

Naive Bayes Performance:
Accuracy : 0.95
Precision : 0.95
Recall : 1.0
F1-Score : 0.97
Confusion Matrix : [[ 97  52]
 [  0 966]]

Message: Free entry in 2 a weekly competition!
Logistic Regression: Spam
Random Forest: Ham
Naive Bayes: Spam

Message: I will meet you at the cafe tomorrow
Logistic Regression: Ham
Random Forest: Ham
Naive Bayes: Ham

Message: Congratulations, you won a free ticket
Logistic Regression: Spam
Random Forest: Ham
Naive Bayes: Ham
